# Estimation legs E1 to E4 (CPU)

Everything here reads the cache the build notebook produced and runs on CPU. The
cost is dominated by the number of replicates, so a first pass at twenty replicates
answers whether the pipeline works, and the registered run at two hundred answers
the paper's questions.

Session settings: **no accelerator**, internet **on**.

Attach the build output as a dataset, plus Provo and SUBTLEX again, since the
reading-time leg and the substantive nulls read the corpus directly.

In [ ]:
!pip -q install pyyaml statsmodels

In [ ]:
!git clone -q https://github.com/garyzhang1006/lossy-context.git /kaggle/working/lossy-context
!pip -q install -e /kaggle/working/lossy-context

In [ ]:
import json, os, subprocess
from pathlib import Path

BUILD     = '/kaggle/input/lcsa-build'
PROVO_DIR = '/kaggle/input/provo-corpus'
SUBTLEX   = '/kaggle/input/subtlex-us/SUBTLEXusfrequencyabove1.csv'
OUT       = '/kaggle/working/artifacts'

CACHE      = f'{BUILD}/cache.npz'
TARGETS    = f'{BUILD}/targets.csv'
CANDIDATES = f'{BUILD}/candidates.json'

# Replicate counts. The registered run is 200 for both; 20 is the smoke setting.
N_REP, N_BOOT = 200, 200

for p in (CACHE, TARGETS, CANDIDATES, PROVO_DIR):
    assert os.path.exists(p), f'not found: {p}'

def run(cmd):
    print(' '.join(cmd))
    r = subprocess.run(cmd)
    if r.returncode != 0:
        raise SystemExit(f'command failed with status {r.returncode}')

## Check the install

Same self-test as the build notebook, and the same rule: if it does not pass, stop.

In [ ]:
!lcsa selftest --out /kaggle/working/selftest

## E1: exactness, sensitivity and the residual fractions

This leg verifies that the Abel form and the atom form of the marginal agree to
machine precision and that the analytic score matches a finite difference, then
reports the residual fraction `||h_perp|| / ||h||` for the human data under each
estimator. That fraction is the empirical weight of the paper and it depends on no
fit of the decay parameter, so it is the first number worth reading.

In [ ]:
run(['lcsa', 'e1', '--cache', CACHE, '--out', OUT])

In [ ]:
s = json.load(open(f'{OUT}/e1_summary.json'))
print(json.dumps(s, indent=2)[:2000])

## E2: recovery ladder and the identification ceiling

Eight readers generated by actual graded truncation at known half-distances are
fitted back, which measures coverage and locates the depth beyond which Provo's
context lengths and response counts stop distinguishing kernels. Registered
prediction 9 puts that ceiling between twelve and thirty words.

In [ ]:
run(['lcsa', 'e2', '--cache', CACHE, '--out', OUT, '--n-rep', str(N_REP)])

In [ ]:
import pandas as pd
print(pd.read_csv(f'{OUT}/e2_ladder.csv').to_string(index=False))

## E3: the zero-decay nulls, then the human fit

The lexical null needs only the cache. The topic and order nulls are built from the
reference model's own behaviour on averaged or shuffled context, so they load the
checkpoint again; on CPU that is slow but it happens once. Drop them with
`--nulls lex` if the session is short.

Registered prediction 1 says both floors reject at most 0.10 under the
cluster-robust test. If they do not, gate G5 voids everything downstream, and the
artifact records that rather than hiding it.

In [ ]:
run(['lcsa', 'e3', '--cache', CACHE, '--out', OUT,
     '--nulls', 'lex,topic,order',
     '--provo-dir', PROVO_DIR, '--targets', TARGETS, '--candidates', CANDIDATES,
     '--n-rep', str(N_REP), '--n-boot', str(N_BOOT)])

In [ ]:
print(pd.read_csv(f'{OUT}/e3_rejection_rates.csv').to_string(index=False))

## E4: the context-limitation sweep and reading times

The sweep asks which truncation window makes reference surprisal predict gaze
duration best, which is the measurement registered prediction 8 says is unstable
across zero-decay references. A target with no eye-tracking record stays missing and
is dropped rather than imputed.

In [ ]:
run(['lcsa', 'e4', '--cache', CACHE, '--out', OUT,
     '--provo-dir', PROVO_DIR, '--targets', TARGETS, '--subtlex', SUBTLEX,
     '--n-boot', str(N_BOOT)])

In [ ]:
print(pd.read_csv(f'{OUT}/e4_sweep_curves.csv').to_string(index=False))

## Everything that was written

These files are what the paper's tables are generated from. Save the directory as a
dataset so the numbers survive the session.

In [ ]:
for p in sorted(Path(OUT).rglob('*')):
    if p.is_file():
        print(f'{str(p.relative_to(OUT)):36s} {p.stat().st_size / 1e3:8.1f} kB')